# Authn vs authz: JWTs, RBAC, and the broken-object hole

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 26 §26.2 · type: walkthrough*

**One-line promise:** secure a FastAPI service with bearer JWTs (authn as a `Depends`), gate an admin route with a role claim (authz/RBAC), then reproduce — and close — the OWASP broken-object-level-authorization hole, all offline with no live IdP.

## 🧠 Why this matters

Two questions get constantly confused, and you must answer both on *every* request. **Authentication** (authn) asks *who are you?* — a bearer token you verify. **Authorization** (authz) asks *what are you allowed to do?* — a role check, and, the part people forget, an *ownership* check on the specific resource.

The single most dangerous API bug is the one where authn passes but authz is missing: the endpoint knows you're a logged-in user, then hands you someone else's record because you changed an id in the URL. That's **broken object-level authorization (BOLA)** — perennially #1 on the OWASP API Top 10. We build the auth, reproduce the breach in three lines, then close it by centralizing the check so it's the default, not per-endpoint memory.

## Objectives & prereqs

**By the end you can:**
- Mint and verify an HS256 JWT (here with the stdlib so the notebook stays dependency-free) and expose authn as a `current_user` dependency that raises `401` on a bad or expired token.
- Trace where that token comes from in an OAuth2 / OIDC authorization-code flow — as a diagram plus a *mocked* token exchange (no live IdP).
- Enforce **RBAC** so an admin-only route returns `403` for non-admins.
- Spot and fix the **BOLA** hole by centralizing the ownership/tenant check.

**Prereqs:** Ch 25 (FastAPI, `Depends`) and Ch 24 (`401` vs `403` semantics). No API key, no IdP, no network — everything is signed and verified locally.

In [ ]:
# --- Setup: imports, env, and the MOCK switch ---------------------------------
# stdlib only (+ python-dotenv & fastapi from requirements.txt). No network is used.
import os
import json
import time
import base64
import hmac
import hashlib
import pathlib

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Offline by design: JWTs are signed/verified locally and the OAuth2 exchange is mocked.
# MOCK=0 would only matter if you wired a real IdP; the lesson runs fully in MOCK=1.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# The signing secret comes from the environment ONLY -- never hardcode it. We fall back to
# a clearly-fake dev secret so the notebook runs with no setup; a real service would fail
# fast if this were unset.
JWT_SECRET = os.getenv("JWT_SECRET", "dev-only-not-a-real-secret")

DATA = pathlib.Path("data")
fixture = json.loads((DATA / "users.json").read_text())
USERS = {u["id"]: u for u in fixture["users"]}
DOCS = {d["id"]: d for d in fixture["documents"]}

have_env = "JWT_SECRET" in os.environ
print(f"MOCK mode: {MOCK}  | secret from env: {have_env} (dev fallback otherwise)")
print("users :", ", ".join(USERS))
print("docs  :", ", ".join(d["id"] + "(owner=" + d["owner"] + ")" for d in DOCS.values()))

## 1 · A JWT, demystified (§26.2)

A JWT is three base64url segments joined by dots: `header.payload.signature`. The header and payload are *not encrypted* — just encoded — so never put secrets in a JWT. The signature is an HMAC over `header.payload` using your secret; that's what makes the token unforgeable. We implement sign/verify in ~15 lines of stdlib so you can see exactly what the `jwt` (PyJWT) library does for you.

In [ ]:
# A minimal HS256 JWT, by hand, so nothing is magic. In production you'd use PyJWT:
#     import jwt; jwt.encode(claims, secret, algorithm="HS256")  # book §26.2
# but the bytes are identical -- a signed, base64url-encoded header.payload.signature.

def _b64url(raw: bytes) -> str:
    return base64.urlsafe_b64encode(raw).rstrip(b"=").decode()

def _b64url_decode(seg: str) -> bytes:
    return base64.urlsafe_b64decode(seg + "=" * (-len(seg) % 4))

def jwt_encode(claims: dict, secret: str) -> str:
    header = {"alg": "HS256", "typ": "JWT"}
    h = _b64url(json.dumps(header, separators=(",", ":")).encode())
    p = _b64url(json.dumps(claims, separators=(",", ":")).encode())
    signing_input = f"{h}.{p}".encode()
    sig = hmac.new(secret.encode(), signing_input, hashlib.sha256).digest()
    return f"{h}.{p}.{_b64url(sig)}"

class JWTError(Exception):
    pass

def jwt_decode(token: str, secret: str) -> dict:
    try:
        h, p, s = token.split(".")
    except ValueError:
        raise JWTError("malformed token")
    expected = _b64url(hmac.new(secret.encode(), f"{h}.{p}".encode(), hashlib.sha256).digest())
    # constant-time compare defeats signature-timing attacks
    if not hmac.compare_digest(expected, s):
        raise JWTError("bad signature")
    claims = json.loads(_b64url_decode(p))
    if "exp" in claims and claims["exp"] < time.time():
        raise JWTError("expired token")
    return claims

def mint_token(user_id: str, ttl_seconds: int = 3600) -> str:
    u = USERS[user_id]
    now = int(time.time())
    return jwt_encode({
        "sub": user_id,            # subject = who
        "tenant": u["tenant"],     # tenant for multi-tenant scoping (Ch 28)
        "roles": u["roles"],       # role claims drive RBAC
        "iat": now,
        "exp": now + ttl_seconds,
    }, JWT_SECRET)

alice_token = mint_token("u_alice")
print("alice's token (note the 3 dot-separated segments):")
print(alice_token)
print("\ndecoded claims:", jwt_decode(alice_token, JWT_SECRET))

🔮 **Predict.** We flip a single character in the token's payload and verify it again. Will `jwt_decode` (a) return the tampered claims, (b) raise `bad signature`, or (c) raise `expired token`? Decide before running.

In [ ]:
# Tamper: flip characters in the middle (payload) segment.
h, p, s = alice_token.split(".")
tampered = f"{h}.{p[:-2]}XY.{s}"
try:
    jwt_decode(tampered, JWT_SECRET)
    print("accepted (!!) -- this would be a forgery vulnerability")
except JWTError as e:
    print("rejected:", e)

**What you just saw.** Any edit to header or payload changes the bytes the HMAC is computed over, so the signature no longer matches — `bad signature`. That's the whole security model: you can *read* a JWT without the secret, but you can't *alter* one.

## 2 · Authn as a `Depends` — the book's `HTTPBearer` shape (🔧 Build)

In FastAPI you express authentication as a dependency: a `current_user` that pulls the bearer token off the `Authorization` header, decodes it, and raises `401` on failure. Every protected route just declares `user = Depends(current_user)`. This is the §26.2 shape:

```python
bearer = HTTPBearer()

async def current_user(token = Depends(bearer)) -> User:
    try:
        claims = jwt.decode(token.credentials, settings.jwt_secret, algorithms=["HS256"])
    except jwt.PyJWTError:
        raise HTTPException(401, "invalid token")
    return await load_user(claims["sub"])
```

We build the same thing against a real FastAPI app and exercise it with the in-process `TestClient` — no server, no network.

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Header, Path
from fastapi.testclient import TestClient

app = FastAPI(title="Agent API", version="1.0.0")

def current_user(authorization: str = Header(default="")) -> dict:
    """Authn dependency: verify the bearer JWT or raise 401. Mirrors the book's
    HTTPBearer + jwt.decode shape, using our stdlib jwt_decode."""
    if not authorization.startswith("Bearer "):
        raise HTTPException(status_code=401, detail="missing bearer token")
    token = authorization.removeprefix("Bearer ")
    try:
        claims = jwt_decode(token, JWT_SECRET)
    except JWTError as e:
        raise HTTPException(status_code=401, detail=f"invalid token: {e}")
    user = USERS.get(claims["sub"])
    if user is None:
        raise HTTPException(status_code=401, detail="unknown subject")
    # carry the verified claims (roles, tenant) alongside the user record
    return {**user, "claims": claims}

@app.get("/me")
def me(user: dict = Depends(current_user)):   # authn as a dependency
    return {"id": user["id"], "tenant": user["tenant"], "roles": user["claims"]["roles"]}

client = TestClient(app)

print("no token      ->", client.get("/me").status_code, "(expect 401)")
print("garbage token ->", client.get("/me", headers={"Authorization": "Bearer not.a.jwt"}).status_code, "(expect 401)")
ok = client.get("/me", headers={"Authorization": f"Bearer {alice_token}"})
print("alice's token ->", ok.status_code, ok.json())

**What you just saw.** One dependency centralizes verification. A missing or invalid token never reaches your handler — it stops at `401`. Note `401` means *"I don't know who you are"*; we'll meet its sibling `403` (*"I know who you are; you may not"*) next.

## 3 · Where does the token come from? OAuth2 / OIDC (mocked)

For "log in with Google/Okta/Entra," your app never sees the password. The **authorization-code flow** runs like this:

```
 Browser            Your app (client)         Identity Provider (IdP)
   |  click 'log in'      |                           |
   |-------------------->  | redirect to IdP --------->|
   |                      |        user authenticates |
   | <----------------------- redirect back w/ CODE --|
   |  ?code=abc           |                           |
   |-------------------->  | exchange CODE + secret -->|
   |                      | <-------- id_token (JWT) --|
   |                      |  verify & start session   |
```

The IdP returns an **id_token** — itself a signed JWT — which is the token your `current_user` then verifies on each call. We mock the final exchange so you can see the shape without a live IdP.

In [ ]:
def mock_oauth_exchange(auth_code: str) -> dict:
    """Stand-in for POST /token at the IdP. A real client sends the code + client
    secret over TLS and receives tokens; here we mint one locally for 'u_carol'."""
    if MOCK:
        # The IdP would verify the code; we just map a known code to a known user.
        assert auth_code == "mock-auth-code-xyz", "unexpected authorization code"
        return {
            "access_token": mint_token("u_carol"),
            "token_type": "Bearer",
            "expires_in": 3600,
        }
    raise RuntimeError("live IdP path not configured; wire an OIDC client to use MOCK=0")

tokens = mock_oauth_exchange("mock-auth-code-xyz")
shown = {k: (v[:24] + "..." if k == "access_token" else v) for k, v in tokens.items()}
print("exchanged code for tokens:", shown)
# That access_token is exactly what your API's current_user() verifies:
who = client.get("/me", headers={"Authorization": f"Bearer {tokens['access_token']}"})
print("/me with the IdP-issued token ->", who.json())

## 4 · RBAC: a role claim gates an admin route (🔧 Build)

Authorization, first cut: **role-based access control**. The token already carries a `roles` claim; an admin-only route checks for `"admin"` and returns `403` otherwise. We express *this* as a dependency too, so the policy lives in one place.

In [ ]:
def require_role(role: str):
    """Authz dependency factory: 403 unless the verified user has `role`."""
    def checker(user: dict = Depends(current_user)) -> dict:
        if role not in user["claims"]["roles"]:
            raise HTTPException(status_code=403, detail=f"requires role: {role}")
        return user
    return checker

@app.get("/admin/metrics")
def admin_metrics(user: dict = Depends(require_role("admin"))):
    return {"ok": True, "viewer": user["id"]}

carol_token = tokens["access_token"]  # carol is an admin (see fixture)
h_alice = {"Authorization": f"Bearer {alice_token}"}
h_carol = {"Authorization": f"Bearer {carol_token}"}
print("alice (user)  ->", client.get("/admin/metrics", headers=h_alice).status_code, "(expect 403)")
print("carol (admin) ->", client.get("/admin/metrics", headers=h_carol).status_code, "(expect 200)")

**What you just saw.** Same authenticated users, different *authorization* outcomes. `401` vs `403` is not pedantry: `401` tells a client "re-authenticate," `403` tells it "don't bother, you lack permission." Returning the wrong one sends clients down the wrong recovery path.

## 5 · ⚠️ The dangerous one: broken object-level authorization (BOLA)

Here is the bug that breaches real companies. We add a `GET /documents/{doc_id}` that authenticates the user — and then returns the document. It *feels* secure: you need a valid token. But it never checks whether *this* user owns *that* document.

🔮 **Predict.** Alice is authenticated. `doc_2` belongs to **Bob**. What status and body does `GET /documents/doc_2` return with Alice's token? Decide before running.

In [ ]:
# THE VULNERABLE VERSION -- authn present, object-level authz MISSING.
@app.get("/documents/{doc_id}")
def get_document_vulnerable(doc_id: str = Path(...), user: dict = Depends(current_user)):
    doc = DOCS.get(doc_id)
    if doc is None:
        raise HTTPException(status_code=404, detail="not found")
    return doc   # <-- no ownership check: returns ANY doc to ANY logged-in user

# Alice walks Bob's id by editing the URL -- the classic IDOR/BOLA attack.
resp = client.get("/documents/doc_2", headers=h_alice)
print("alice reads Bob's doc_2 ->", resp.status_code)
print(resp.json())

**What you just saw.** `200 OK` — Alice just read Bob's document by changing two characters in the URL. Authentication was never the problem; the *missing object-level authorization* check was. With `tenant` in play, the same hole leaks data *across customers* — the kind of incident that ends up in the press.

### The fix: centralize the ownership/tenant check (🔧 Build)

The cure is not "remember to add an `if` to every endpoint" — that's how the bug recurs. Make the authorized lookup the *only* way to fetch a resource: a dependency that loads the object **and** asserts the caller owns it (and shares its tenant). Now forgetting the check is impossible because there's no unchecked path.

In [ ]:
def authorized_document(doc_id: str = Path(...), user: dict = Depends(current_user)) -> dict:
    """Load-and-authorize in ONE place. 404 if absent; deny (also 404) if not owner or
    wrong tenant. Returning 404 for someone else's resource avoids leaking its existence."""
    doc = DOCS.get(doc_id)
    if doc is None:
        raise HTTPException(status_code=404, detail="not found")
    same_tenant = doc["tenant"] == user["tenant"]
    is_owner = doc["owner"] == user["id"]
    is_admin = "admin" in user["claims"]["roles"]
    if not (same_tenant and (is_owner or is_admin)):
        raise HTTPException(status_code=404, detail="not found")  # deny == 404, hide existence
    return doc

@app.get("/v2/documents/{doc_id}")
def get_document_safe(doc: dict = Depends(authorized_document)):
    return doc   # the handler is trivial; the policy lives in the dependency

a = client.get("/v2/documents/doc_2", headers=h_alice)
print("alice -> Bob's doc_2 :", a.status_code, "(expect 404 -- denied, existence hidden)")
b = client.get("/v2/documents/doc_1", headers=h_alice)
print("alice -> her doc_1   :", b.status_code, b.json().get("title"))
cc = client.get("/v2/documents/doc_3", headers=h_carol)
print("carol -> globex doc_3:", cc.status_code, cc.json().get("title"))

**What you just saw.** Same attack, now `404` — and Alice still reads her own document, Carol still reads hers. By making `authorized_document` the single door to a document, the ownership and tenant checks are the *default*, not a thing each endpoint must remember. We return `404` rather than `403` for another tenant's id so we don't even confirm it exists.

## 🎯 Senior lens

The big judgment call in auth is **JWT vs server-side sessions**, and it comes down to *revocation*. A JWT is **stateless** — fast, no lookup, scales horizontally — but that's exactly why it's hard to kill: a stolen token is valid until it expires, because nothing checks a server record. Sessions invert the trade: a central store means you can revoke instantly (logout-everywhere, ban a user mid-incident), at the cost of a lookup per request and shared state across instances.

The senior move is to pick by your revocation needs and often combine: short-lived access JWTs (minutes) plus a refresh token you *can* revoke; or JWTs with a small cached denylist of `jti`s checked on sensitive routes. And whichever you choose, enforce authz in *one* place — the centralized ownership/tenant dependency you just built — because BOLA is a process failure (a forgotten check), not a clever exploit.

## Recap

- **Authn (`who?`) ≠ authz (`what may you do?`)**; enforce both, every request. `401` = re-authenticate; `403` = you lack permission.
- A JWT is a *signed, readable* token: you can decode it without the secret, but not forge it. Never put secrets in the payload.
- Express authn as a `current_user` **dependency** and RBAC as a `require_role(...)` dependency — policy in one place, trivial handlers.
- OAuth2/OIDC's authorization-code flow is where the JWT comes from in "log in with…"; your API just verifies the resulting id/access token.
- **BOLA** is the #1 API risk: authn present, object-level authz missing. Centralize the ownership/tenant check so the unchecked path doesn't exist; deny with `404` to hide existence.

## Exercises

Predict the result before running each.

1. **Expiry.** Mint a token with `ttl_seconds=1`, wait two seconds, and call `/me`. What status, and which `JWTError` fires inside `current_user`? Then mint with `ttl_seconds=-10` and explain the difference (if any).
2. **Wrong secret = forgery defense.** Re-decode `alice_token` with a *different* `JWT_SECRET`. Which exception? Tie this back to why the secret must come from the environment and never ship in code.
3. **Tenant isolation.** Add a fourth user in `globex` and a document they own, then verify Alice (acme) gets `404` on it via `/v2/documents/...`. Why is `404` a better deny code than `403` here?
4. **ABAC tweak.** Extend `authorized_document` so a `"viewer"` role may *read* any doc in its own tenant but never another tenant's. Predict the matrix of outcomes for alice/bob/carol before testing.

In [ ]:
# Exercise 1 -- your code here


In [ ]:
# Exercise 2 -- your code here


In [ ]:
# Exercise 3 -- your code here


In [ ]:
# Exercise 4 -- your code here


## Next

- ➡️ **Next notebook:** [`26-02-rate-limiting-quotas-errors-pagination.ipynb`](./26-02-rate-limiting-quotas-errors-pagination.ipynb) — make the API defend itself: per-tenant rate limits with `429` + `Retry-After`, a structured error envelope, and cursor pagination.
- 📓 **Book:** §26.2 (authentication & authorization) — the `HTTPBearer` / `jwt.decode` shape and the BOLA pitfall.
- 🧱 **Template:** the auth dependency becomes a production default in [`templates/fastapi-agent-service/`](../../../templates/fastapi-agent-service/).
- 🏗️ **Capstone:** wraps `capstone/app/` with auth + per-tenant scoping (checkpoint `ch26-enterprise-api`).